![HydroCycle](images/hydro_5cycle.jpg)

# Retrieve and Analyze Hydrology data for a watershed of interest

To make predictions for reservoir operations, water supply, flood control, etc, we need to collect data to train/calibrate hydrologic models. This includes streamflow, current environmental conditions (e.g., snow water equivalent), and future weather predictions. This exercise will build on the previous SNOTEL module to work towards building a hydrologic module.

Need to find a station? Use the [USGS NWIS mapper system](https://apps.usgs.gov/nwismapper/)


Click the link and explore!

## Create a HydroDataFrame

We want to create one data frame containing streamflow, meteological information, and SNOTEL for our period of record

In [1]:
from pynhd import NLDI
import geopandas as gpd
import pandas as pd
from supporting_scripts import getData, SNOTEL_Analyzer, dataprocessing, mapping
from shapely.geometry import box, Polygon
import os
import datetime
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
station_id = "11274790" # NWIS id for Tuolumne river at the mouth of Hetch Hetchy Reservoir
basinname = 'TuolumneRiverBasin'

## Load data

We need to load the saved data into our script to process and combine.

We will start with the SNOTEL data

In [3]:
#load snotel data
unprocessed_SNOTEL = {}
#read all files in the following path into the dictionary
path = 'files/SNOTEL'
for filename in os.listdir(path):
    if filename.endswith('.csv'):
        #select the name of the file between the _ and _
        name = filename.split('_')[1] 
        unprocessed_SNOTEL[name] = pd.read_csv(os.path.join(path, filename))
        #make the date a datetime object and set to the index
        unprocessed_SNOTEL[name]['Date'] = pd.to_datetime(unprocessed_SNOTEL[name]['Date'])
        unprocessed_SNOTEL[name].set_index('Date', inplace=True)
        #rename the Snow Water Equivalent (m) Start of Day Values to SWE_cm
        unprocessed_SNOTEL[name].rename(columns={'Snow Water Equivalent (m) Start of Day Values': f"{name}_SWE_cm"}, inplace=True)
        #convert SWE_m to cm
        unprocessed_SNOTEL[name][f"{name}_SWE_cm"] = unprocessed_SNOTEL[name][f"{name}_SWE_cm"] * 100
        #remove the Water_Year column
        unprocessed_SNOTEL[name].drop(columns=['Water_Year'], inplace=True)
        #we need to know how many obs for each DF, print the df name, its length, and the start/end dates
        print(f"{name}: {len(unprocessed_SNOTEL[name])} start date: {unprocessed_SNOTEL[name].index.min()} end date: {unprocessed_SNOTEL[name].index.max()}")
    



386: 16068 start date: 1978-10-01 00:00:00 end date: 2022-09-27 00:00:00
387: 12940 start date: 1990-10-01 00:00:00 end date: 2026-03-05 00:00:00
629: 17323 start date: 1978-10-01 00:00:00 end date: 2026-03-05 00:00:00
632: 14456 start date: 1986-08-07 00:00:00 end date: 2026-03-05 00:00:00
713: 16592 start date: 1980-10-01 00:00:00 end date: 2026-03-05 00:00:00
780: 14429 start date: 1986-09-03 00:00:00 end date: 2026-03-05 00:00:00
DAN: 6195 start date: 2004-10-01 00:00:00 end date: 2021-09-16 00:00:00
SLI: 5892 start date: 2005-10-01 00:00:00 end date: 2021-11-17 00:00:00
TES: 629 start date: 2005-03-01 00:00:00 end date: 2006-11-19 00:00:00
TUM: 6257 start date: 2004-10-01 00:00:00 end date: 2021-11-17 00:00:00


In [4]:
#The TES site is missing many values and will not be useful for our analysis, remove it
unprocessed_SNOTEL.pop('TES', None)

#The site with the latest start date will guide the rest
latest_start_date = max([df.index.min() for df in unprocessed_SNOTEL.values()])

#The site with the earliest end date will guide the rest
soonest_end_date = min([df.index.max() for df in unprocessed_SNOTEL.values()])
for key in unprocessed_SNOTEL.keys():
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index >= latest_start_date]
    unprocessed_SNOTEL[key] = unprocessed_SNOTEL[key][unprocessed_SNOTEL[key].index <= soonest_end_date]

#merge all dictionary dataframes into one larger dataframe
SNOTEL_df = pd.concat(unprocessed_SNOTEL.values(), axis=1)
#set the date index to be the index of the first dataframe in the dictionary

SNOTEL_df.head()

,386_SWE_cm,387_SWE_cm,629_SWE_cm,632_SWE_cm,713_SWE_cm,780_SWE_cm,DAN_SWE_cm,SLI_SWE_cm,TUM_SWE_cm
Date,,,,,,,,,
2005-10-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2005-10-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2005-10-03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2005-10-04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2005-10-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Load the meteorologogical data from Daymet and NLDAS

In [5]:
#Read the data from PyDayMet 
PyDayMet_df = pd.read_csv(f"files/PyDayMet/PyDayMet_{station_id}.csv")
#set the date column to be a datetime object and set it to the index
PyDayMet_df['Date'] = pd.to_datetime(PyDayMet_df['Date'])
PyDayMet_df.set_index('Date', inplace=True)
PyDayMet_df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'files/PyDayMet/PyDayMet_11274790.csv'

In [ ]:
#Read the data from NLDAS 
NLDAS_df = pd.read_csv(f"files/NLDAS/NLDAS_{station_id}.csv")
#set the date column to be a datetime object and set it to the index
NLDAS_df['Date'] = pd.to_datetime(NLDAS_df['Date'])
NLDAS_df.set_index('Date', inplace=True)
NLDAS_df.head()

## Load the streamflow data

In [ ]:
streamflow_df = pd.read_csv(f"files/NWIS/streamflow_{station_id}.csv")
#set the date column to be a datetime object and set it to the index
streamflow_df['Date'] = pd.to_datetime(streamflow_df['Date'])
streamflow_df.set_index('Date', inplace=True)

streamflow_df.head()

## Load our catchment information

In [ ]:
basin_info = pd.read_csv(f"files/basin_info/basin_info_{station_id}.csv")
basin_info.head()

## Now we can merge into one dataframe

In [ ]:
#find the latest start date and the earliest end date for SNOTEL_df, met_df, cleaned
begin_date = max([df.index.min() for df in [SNOTEL_df, PyDayMet_df, streamflow_df, NLDAS_df]]) 
end_date = min([df.index.max() for df in [SNOTEL_df, PyDayMet_df, streamflow_df, NLDAS_df]]) 

#clip each dataframe to have the same begin and end dates
SNOTEL_df = SNOTEL_df[(SNOTEL_df.index >= begin_date) & (SNOTEL_df.index <= end_date)]
PyDayMet_df = PyDayMet_df[(PyDayMet_df.index >= begin_date) & (PyDayMet_df.index <= end_date)]
streamflow_df = streamflow_df[(streamflow_df.index >= begin_date) & (streamflow_df.index <= end_date)]
NLDAS_df = NLDAS_df[(NLDAS_df.index >= begin_date) & (NLDAS_df.index <= end_date)]

In [ ]:
#merge the SNOTEL_df, met_df, and streamflow dataframes
Hydro_df = pd.concat([SNOTEL_df, PyDayMet_df, NLDAS_df,streamflow_df], axis=1)
#put the site_no column, second to last, and streamfow column, last column, as the first two columns in the dataframe
cols = Hydro_df.columns.tolist()
cols = cols[-2:] + cols[:-2]
Hydro_df = Hydro_df[cols]
Hydro_df.head()

In [ ]:
#all of the NaN values here should be 0, fill them
Hydro_df = Hydro_df.fillna(0)
Hydro_df.head()

In [ ]:
#add in the basin info as columns in the dataframe, repeat the values for each row
for col in basin_info.columns:
    Hydro_df[col] = basin_info[col][0]

Hydro_df.head()

In [ ]:
#take a look around peak SWE to make sure we have snotel values, early season can be tricky to assess
Hydro_df.loc['2019-03-01':'2019-04-01']

In [ ]:
Hydro_df.tail()

## Data Exploration Exercise

Use the combined data frame you created for a **USGS NWIS site other than the example** to explore different data and relationships. Select a single water year (October - September) to explore the following relationships:
* Streamflow and SWE (e.g., Daymet and SWE from Snotel)
* Snowmelt -  SWE and temperature/SW radiation
* Rainfall/runoff response - precipitation and streamflow
* comparison of the NLDAS precipitation and temperature with DayMet precipitation and temperature

In markdown below each figure, write a few sentences describing the relationships and anything that is unexpected. This exercise prepares you for HW#2.
